# Data Cleaning

This chapter includes the following:
* How to read and clean data
* Creating new indicators

In this chapter, a method is proposed to simplify the computations involved in processing large datasets. Although advancements in processing power may eventually eliminate the need for such methods, the current volume of data makes it impractical to load all of it into memory at once. Therefore, it is recommended to process the data incrementally by reading it in chunks. For each chunk, the number of observations for each metropolitan area is calculated. After processing all chunks, the results are aggregated to determine the total number of observations for each metropolitan area across the entire dataset. By sorting these results, the top 20 metropolitan areas can be identified without requiring extensive memory resources. This approach provides the necessary indicators for further filtering.

In the next phase, the data is again processed in chunks. During each iteration, specific rows are filtered based on the criteria established, and new columns containing the required information are generated. The filtered data from each chunk is then compiled into a single output file, facilitating further analysis while managing memory constraints effectively.

# import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')

---
# Section I. Downloading Data (Optional)
NOTE: The raw HMDA file is not included in this repository (see `../data/README.md`). If you have already downloaded it, skip `Section I` and place `2020_lar.txt` in `../data/raw/`.

## download and unzip data

In [ ]:
!mkdir -p ../data/raw
!wget -P ../data/raw https://s3.amazonaws.com/cfpb-hmda-public/prod/dynamic-data/2020/2020_lar.zip
!unzip ../data/raw/2020_lar.zip -d ../data/raw

---

# Section II. Reading Required Data
# define file paths

In [ ]:
import os

# All paths are relative to this notebook's folder (<project>/code)
input_file = '../data/raw/2020_lar.txt'                  # raw HMDA 2020 LAR file (not included; see ../data/README.md)
msa_file = '../data/processed/msa_data.csv'              # originations by MSA
filtered_file = '../data/processed/filtered_data.csv'    # sample after the restrictions in Question 1
cleaned_file = '../data/processed/cleaned_data.csv'      # analysis sample
os.makedirs('../data/processed', exist_ok=True)

# Processes MSA of a large input CSV file in chunks

In [ ]:
import pandas as pd

def process_large_file(input_file, msa_file, chunk_size=100000):
    """
    Processes a large input CSV file in chunks, filters relevant columns, aggregates data,
    and writes the aggregated results to an output CSV file.

    Args:
        input_file (str): The path to the input CSV file.
        msa_file (str): The path to the output CSV file where the results are written.
        chunk_size (int): The number of rows per chunk to be processed. Default is 100,000.
    """
    # Initialize round counter
    round_counter = 0

    # Open the output file in append mode
    with open(msa_file, 'a') as f_out:
        # Read the input file in chunks
        for chunk in pd.read_csv(input_file, chunksize=chunk_size, delimiter='|', dtype=str):
            # Increment the counter to track how many chunks have been processed
            round_counter += 1

            # Filter the data (only keep the "derived_msa_md" and "loan_amount" columns)
            chunk = chunk[["derived_msa_md", "loan_amount"]].copy()  # Ensure a copy is made to avoid SettingWithCopyWarning

            # Convert "loan_amount" column to numeric, replace non-numeric values with 0
            chunk.loc[:, 'loan_amount'] = pd.to_numeric(chunk['loan_amount'], errors='coerce').fillna(0)

            # Aggregate the data by "derived_msa_md" column
            result = chunk.groupby('derived_msa_md').agg(
                total_loan_amount=('loan_amount', 'sum'),
                count=('derived_msa_md', 'size')
            ).reset_index()

            # Append the aggregated result to the output file
            result.to_csv(f_out, header=f_out.tell() == 0, index=False)

            # Print the progress every 10 chunks
            if round_counter % 10 == 0:
                print(f"Processed chunk number {round_counter}")


process_large_file(input_file, msa_file)


Processed chunk number 10
Processed chunk number 20
Processed chunk number 30
Processed chunk number 40
Processed chunk number 50
Processed chunk number 60
Processed chunk number 70
Processed chunk number 80
Processed chunk number 90
Processed chunk number 100
Processed chunk number 110
Processed chunk number 120
Processed chunk number 130
Processed chunk number 140
Processed chunk number 150
Processed chunk number 160
Processed chunk number 170
Processed chunk number 180
Processed chunk number 190
Processed chunk number 200
Processed chunk number 210
Processed chunk number 220
Processed chunk number 230
Processed chunk number 240
Processed chunk number 250


# consolidate chunk result

In [ ]:
def aggregate_final_results(msa_file):
    """
    Aggregates the results from the output CSV file to compute the final total loan amount and count.

    Args:
        msa_file (str): The path to the output CSV file to be aggregated.
    """
    # Read the output file
    df = pd.read_csv(msa_file)

    # Aggregate the data by "derived_msa_md" column
    result = df.groupby('derived_msa_md').agg(
        total_loan_amount=('total_loan_amount', 'sum'),
        count=('count', 'sum')
    ).reset_index()

    return result

final_result = aggregate_final_results(msa_file)

# find top 20 metropolitan areas

In [ ]:
def find_top20_metro_areas(result):
    """
    Finds the top 20 metropolitan areas by count.

    Args:
        result (DataFrame): The aggregated result DataFrame.

    Returns:
        Series: The "derived_msa_md" values of the top 20 metropolitan areas by count.
    """
    # Find the top 20 metropolitan areas by count
    top20 = result.nlargest(20, 'count')["derived_msa_md"]

    return top20

top20 = find_top20_metro_areas(final_result)

# filtering function
 Restrict the sample to: top 20 metropolitan areas by mortgage origination; conventional, first-lien, mortgages originated for purchases of owner-occupied single-family homes with closed-end credit.

In [ ]:
def filtering(df, top20):
    """
    Filters the DataFrame based on specified criteria, including restricting to
    the top 20 metropolitan areas, conventional, first-lien, mortgages
    originated for purchases of owner-occupied single-family homes with
    closed-end credit.

    Args:
        df (DataFrame): The DataFrame to be filtered.
        top20 (DataFrame): The DataFrame of top 20 metropolitan areas by mortgage origination.

    Returns:
        DataFrame: Filtered DataFrame containing relevant columns.
    """
    # Restrict sample to top 20 metropolitan areas by mortgage origination
    filtered_data = df[df['derived_msa_md'].isin(top20)]

    # Filter for conventional, first-lien, owner-occupied single-family home mortgages
    filtered_data = filtered_data[(filtered_data['loan_type'] == 1) &    # Conventional
                                  (filtered_data['lien_status'] == 1) &  # First lien
                                  (filtered_data['derived_loan_product_type'] == 'Conventional:First Lien') &
                                  (filtered_data['loan_purpose'] == 1) &  # Home purchase
                                  (filtered_data['occupancy_type'] == 1) &  # Owner-occupied
                                  (filtered_data['open_end_line_of_credit'] == 2) &  # Closed-end Credit
                                  ((filtered_data['derived_dwelling_category'] == 'Single Family (1-4 Units):Site-Built') |
                                  (filtered_data['derived_dwelling_category'] == 'Single Family (1-4 Units):Manufactured')) # Single-family
                                  ]

    # Convert data types
    filtered_data['total_loan_costs'] = pd.to_numeric(filtered_data['total_loan_costs'], errors='coerce')
    filtered_data['loan_amount'] = pd.to_numeric(filtered_data['loan_amount'], errors='coerce')
    filtered_data['rate_spread'] = pd.to_numeric(filtered_data['rate_spread'], errors='coerce')
    filtered_data['combined_loan_to_value_ratio'] = pd.to_numeric(filtered_data['combined_loan_to_value_ratio'], errors='coerce')

    # Create a categorical variable for borrower race
    filtered_data['race'] = np.nan  # Explicitly set as dtype 'object' to allow mixed types
    filtered_data = filtered_data.astype({'race': 'object'})

    filtered_data.loc[(filtered_data['derived_race'] == 'White') & (filtered_data['derived_ethnicity'] != 'Hispanic or Latino'), 'race'] = 'White non-Hispanic'
    filtered_data.loc[filtered_data['derived_race'] == 'Asian', 'race'] = 'Asian'
    filtered_data.loc[filtered_data['derived_race'] == 'Black or African American', 'race'] = 'Black'
    filtered_data.loc[filtered_data['derived_ethnicity'] == 'Hispanic or Latino', 'race'] = 'Hispanic/Latino'
    filtered_data = filtered_data[filtered_data['race'].notna()]

    # Create 'market' variable as MSA x Race
    filtered_data['market'] = filtered_data['derived_msa_md'].astype(str) + '_' + filtered_data['race']

    # Create 'fee_share' variable
    filtered_data['fee_share'] = filtered_data['total_loan_costs'] / filtered_data['loan_amount']

    # Create 'rejected' variable
    filtered_data['rejected'] = (filtered_data['action_taken'] == 3).astype(int)

    # Keep required columns
    filtered_data = filtered_data[["race", "derived_msa_md", "market", "fee_share",  "loan_amount",
                                   "rate_spread", "combined_loan_to_value_ratio", "lei", "rejected",
                                   "applicant_credit_score_type", "income", "applicant_sex", "derived_sex",
                                   "co_applicant_credit_score_type", "lender_credits",
                                   "loan_term", "interest_rate", "total_loan_costs",
                                   "applicant_age_above_62", "debt_to_income_ratio"
                                   ]]
    return filtered_data


# run filtering on input data in chunks

In [ ]:
def process_filtered_file(input_file, filtered_file, top20, chunk_size=100000):
    """
    Processes a large input CSV file in chunks, applies filtering, and writes
    the filtered results to an output CSV file.

    Args:
        input_file (str): The path to the input CSV file.
        filtered_file (str): The path to the output CSV file where the filtered results are written.
        top20 (DataFrame): The DataFrame of top 20 metropolitan areas by mortgage origination.
        chunk_size (int): The number of rows per chunk to be processed. Default is 100,000.
    """
    # Initialize round counter
    round_counter = 0

    # Open the output file in append mode
    with open(filtered_file, 'a') as f_out:
        # Read the input file in chunks
        for chunk in pd.read_csv(input_file, chunksize=chunk_size, delimiter='|'):
            # Increment the counter to track how many chunks have been processed
            round_counter += 1

            # Apply filtering function
            filtered_data = filtering(chunk, top20)

            # Append the filtered chunk to the output file
            filtered_data.to_csv(f_out, header=f_out.tell() == 0, index=False)

            # Print the progress every 10 chunks
            if round_counter % 10 == 0:
                print(f"Processed chunk number {round_counter} with the length of {len(filtered_data)}")

process_filtered_file(input_file, filtered_file, top20)

Processed chunk number 10 with the length of 6103
Processed chunk number 20 with the length of 6075
Processed chunk number 30 with the length of 693
Processed chunk number 40 with the length of 5542
Processed chunk number 50 with the length of 4196
Processed chunk number 60 with the length of 3938
Processed chunk number 70 with the length of 8440
Processed chunk number 80 with the length of 8490
Processed chunk number 90 with the length of 10662
Processed chunk number 100 with the length of 2737
Processed chunk number 110 with the length of 2445
Processed chunk number 120 with the length of 9077
Processed chunk number 130 with the length of 3706
Processed chunk number 140 with the length of 4715
Processed chunk number 150 with the length of 11120
Processed chunk number 160 with the length of 5428
Processed chunk number 170 with the length of 6453
Processed chunk number 180 with the length of 2645
Processed chunk number 190 with the length of 8812
Processed chunk number 200 with the len

# clean filtered data

In [ ]:
def clean_filtered_data(filtered_file):
    """
    Cleans the filtered data by handling missing values, converting data types, and removing outliers.

    Args:
        filtered_file (str): The path to the filtered CSV file to be cleaned.

    Returns:
        DataFrame: Cleaned DataFrame.
    """
    # Read the filtered file
    data = pd.read_csv(filtered_file)

    # Replace 'NA' and None values with NaN
    data.replace("NA", np.nan, inplace=True)
    data.replace(to_replace=[None], value=np.nan, inplace=True)
    data.replace("Exempt", '0', inplace=True)

    # Convert data types
    data["income"] = data["income"].astype("float")
    data["lender_credits"] = data["lender_credits"].astype("float")
    data["loan_term"] = data["loan_term"].astype("float")
    data['interest_rate'] = data['interest_rate'].astype('float')

    # Handle outliers for specified columns
    for col in ["rate_spread", "combined_loan_to_value_ratio", "fee_share"]:
        lower_bound = data[col].quantile(0.005)
        upper_bound = data[col].quantile(0.995)
        data[f"{col}_outlier"] = (
            (data[col] < lower_bound) | (data[col] > upper_bound)
        ).astype(int)

    # Drop rows where any of the outlier columns has a value of 1
    outlier_columns = [
        f"{col}_outlier"
        for col in ["rate_spread", "combined_loan_to_value_ratio", "fee_share"]
    ]
    data = data[(data[outlier_columns].sum(axis=1) == 0)]
    data = data.drop(outlier_columns, axis=1)

    return data

cleaned_data = clean_filtered_data(filtered_file)


# lender_shares

In [ ]:
def calculate_lender_shares(data):
    """
    Calculates the lender shares for non-rejected loans and merges the result
    back to the original DataFrame.

    Args:
        data (DataFrame): The cleaned DataFrame containing mortgage data.

    Returns:
        data: DataFrame with an additional 'lender_share' column.
    """
    # Calculate lender shares for non-rejected loans
    lender_shares = (
        data.query("rejected == 0")
        .groupby(["market", "lei"])
        .size()
        .div(data.query("rejected == 0").groupby("market").size())
        .reset_index(name="lender_  share")
    )

    # Merge lender shares back to the original DataFrame
    data = data.merge(lender_shares, on=["market", "lei"], how="left")

    return data

calculate_lender_shares(cleaned_data)

# save cleaned data

In [ ]:
def save_cleaned_data(data, cleaned_file):
    """
    Saves the cleaned data to a specified CSV file.

    Args:
        data (DataFrame): The cleaned DataFrame to be saved.
        cleaned_file (str): The path to the output CSV file where the cleaned data will be written.
    """
    # Save the cleaned data to the specified file
    with open(cleaned_file, 'w') as f_out:
        data.to_csv(f_out, header=f_out.tell() == 0, index=False)

save_cleaned_data(cleaned_data, cleaned_file)

All needed variables are created, cleaned and stored as `cleaned_file`. $\blacksquare$

---